In [ ]:
# ---- 설치 안내 ----
# Colab이면 한 번만 실행:
!pip -q install ultralytics openpyxl tqdm scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.2 MB/s eta 0:00:00


In [ ]:
# 과정1: warp된 베드 이미지 -> lettuce-seg 1회 -> mask(jsonl) 저장 + mid-data(parquet) 저장
# - 재귀 탐색
# - 병렬 로딩 + 진행률/ETA
# - chunk 단위 중간저장/재시작 가능
# - (중요) 여기서는 slot naming/병합을 하지 않는다. (instance 단위 보존)

import os, re, json, time
from glob import glob
from concurrent.futures import ThreadPoolExecutor, as_completed


def load_images_batch(paths: list, io_workers: int):
    """이미지 로딩 I/O 병렬. (순서 보장)"""
    if not paths:
        return []

    def _read(p):
        try:
            return cv2.imread(p)
        except Exception:
            return None

    out = [None] * len(paths)
    with ThreadPoolExecutor(max_workers=io_workers) as ex:
        futs = {ex.submit(_read, p): i for i, p in enumerate(paths)}
        for fut in as_completed(futs):
            i = futs[fut]
            try:
                out[i] = fut.result()
            except Exception:
                out[i] = None
    return out


import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# =========================
# 0) CFG (여기만 수정)
# =========================
BED_WARP_FOLDER = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/ROI_WARP/step6_warp_images/보정 후"    # warp된 베드 이미지 폴더(하위 재귀)
SCALE_CSV       = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv"    # bed_id별 px_per_mm_x, px_per_mm_y 등이 들어있는 파일
SEG_MODEL_PATH  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/model/results/runs/bed_seg_v1/weights/best.pt"
OUT_DIR         = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce"

# 성능/운영 파라미터
N_WORKERS   = 8
BATCH_SIZE  = 16
IO_WORKERS = 12   # 이미지 로딩 병렬 워커 수(8~16 권장)

# YOLO 가중치 경로(본인 파일로 수정)
# 예) LETTUCE_SEG_WEIGHT = "/content/drive/MyDrive/weights/lettuce_seg.pt"
CHUNK_SIZE = 20000  # instance rows 누적 기준(중간 저장)
BATCH_SIZE = 16      # YOLO batch predict용 (GPU 메모리 보고 8~32 조절)   # instance row 기준

# YOLO 파라미터
PRED_CONF = 0.05
PRED_IOU  = 0.7

# 저장 파일명
MID_PARQUET_PREFIX = "mid_instances_chunk"  # 예: mid_instances_chunk_00000.parquet     # instance 메타+지표
MASK_JSONL_NAME  = "masks_rle.jsonl"           # instance_uid 단위 RLE

In [ ]:
# =========================
# 1) Utils
# =========================
IMG_PATTERNS = ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.png", "**/*.PNG", "**/*.webp"]

def now_hms():
    return time.strftime("%H:%M:%S")

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def list_images_recursive(root_dir: str):
    paths = []
    for pat in IMG_PATTERNS:
        paths.extend(glob(os.path.join(root_dir, pat), recursive=True))
    paths = sorted(list(set(paths)))
    return paths

# 예: .../bed00_20251213_062821_cam2.jpg 같은 파일명에서 bed_id/date 추출
# 너 프로젝트에서 쓰던 규칙에 맞게 여기만 조정하면 됨
BEDDATE_RE = re.compile(r"(bed\d+)_(\d{8})")

def parse_bed_id_date(img_path: str):
    bn = os.path.basename(img_path)
    m = BEDDATE_RE.search(bn)
    if not m:
        return None, None, None
    bed_id = m.group(1)
    date = m.group(2)  # YYYYMMDD
    bed_date = f"{bed_id}_{date}"
    return bed_id, date, bed_date

# -------------------------
# RLE (COCO style)
# -------------------------

def mask_to_rle(mask: np.ndarray):
    """mask: uint8 (0/1) shape (H,W) -> RLE dict (counts as list), for jsonl"""
    # column-major flatten (Fortran order) is common in COCO
    pixels = mask.T.flatten()
    # pad with zeros at both ends
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return {"counts": runs.tolist(), "size": [int(mask.shape[0]), int(mask.shape[1])]}


In [ ]:
# =========================
# 2) Scale map 로드
# =========================

def load_scale_map(scale_csv: str):
    """scale_map.csv 구조(캡쳐 기준):
    image_name, date, cyl_ok, cyl_diam_px, cm_per_px_today, qc_flag, method

    반환:
      scale_map[bed_id] = {"cm_per_px": float}

    주의:
    - bed_id는 image_name에서 (bed\d+)_ 패턴으로 추출
    - 동일 bed_id가 여러 행이면 마지막 행 값으로 덮어씀
      (원하면 date 기준 최신만 남기도록 확장 가능)
    """
    df = pd.read_csv(scale_csv)

    need_cols = ["image_name", "cm_per_px_today"]
    for c in need_cols:
        if c not in df.columns:
            raise ValueError(f"SCALE_CSV에 '{c}' 컬럼이 없음. 현재 컬럼: {list(df.columns)}")

    out = {}
    for _, r in df.iterrows():
        img_name = str(r["image_name"])
        cm_per_px = r["cm_per_px_today"]
        if pd.isna(cm_per_px):
            continue

        m = re.search(r"(bed\d+)_", img_name)
        if not m:
            continue
        bed_id = m.group(1)

        out[bed_id] = {"cm_per_px": float(cm_per_px)}

    return out

# =========================
# 3) 이미지 로딩(병렬)
# =========================

def read_image_bgr(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    return img



<>:13: SyntaxWarning: invalid escape sequence '\d'
<>:13: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-2681301762.py:13: SyntaxWarning: invalid escape sequence '\d'
  - bed_id는 image_name에서 (bed\d+)_ 패턴으로 추출


In [ ]:
# =========================
# 4) YOLO seg -> instance rows 만들기
# =========================

def masked_brightness_mean_from_gray(gray: np.ndarray, mask_u8: np.ndarray):
    """마스크(상추 픽셀) 기반 밝기 평균. 픽셀이 너무 적으면 전체 평균으로 fallback"""
    m = mask_u8 > 0
    if int(m.sum()) < 50:
        return float(gray.mean())
    return float(gray[m].mean())


def masked_blur_score_from_lap(lap: np.ndarray, mask_u8: np.ndarray):
    """사용자 기존 정의: Laplacian variance (mask 픽셀 기반, 픽셀 적으면 전체)"""
    m = mask_u8 > 0
    if int(m.sum()) < 50:
        return float(lap.var())
    return float(lap[m].var())


def compute_basic_shape(mask_u8: np.ndarray):
    """mask_u8: 0/1 uint8"""
    area = int(mask_u8.sum())
    if area <= 0:
        return 0, np.nan, np.nan, np.nan

    cnts, _ = cv2.findContours((mask_u8*255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return area, np.nan, np.nan, np.nan

    c = max(cnts, key=cv2.contourArea)
    peri = float(cv2.arcLength(c, True))
    hull = cv2.convexHull(c)
    hull_area = float(cv2.contourArea(hull)) if hull is not None else np.nan

    solidity = float(area / hull_area) if hull_area and hull_area > 0 else np.nan
    circularity = float((4*np.pi*area)/(peri*peri)) if peri and peri > 0 else np.nan

    return area, peri, solidity, circularity

def bbox_from_mask(mask_u8: np.ndarray):
    ys, xs = np.where(mask_u8 > 0)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    return x1, y1, x2, y2

def touch_border(mask_u8: np.ndarray):
    H, W = mask_u8.shape
    if mask_u8[0,:].any() or mask_u8[H-1,:].any() or mask_u8[:,0].any() or mask_u8[:,W-1].any():
        return 1
    return 0

def process_one_result(r0, img_bgr: np.ndarray, img_path: str, scale_map: dict):
    """model.predict 결과(r0: Results)를 받아 instance rows/masks 생성.
    batch predict에서 사용 (여기서는 model.predict 호출 금지)
    """
    bed_id, date, bed_date = parse_bed_id_date(img_path)
    if bed_id is None:
        return [], []

    H, W = img_bgr.shape[:2]

    # scale 읽기: cm_per_px (cm/px)
    cm_per_px = np.nan
    if isinstance(scale_map, dict) and bed_id in scale_map:
        d = scale_map[bed_id]
        if isinstance(d, dict):
            if "cm_per_px" in d:
                cm_per_px = float(d["cm_per_px"])
            elif "cm_per_px_today" in d:
                cm_per_px = float(d["cm_per_px_today"])

    rows, mask_recs = [], []

    if r0 is None or r0.masks is None or r0.boxes is None:
        return [], []

    masks = r0.masks.data.cpu().numpy()   # (N,h,w)
    confs = r0.boxes.conf.cpu().numpy()   # (N,)

    # 이미지당 1회 계산(속도 개선)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)

    n = masks.shape[0]
    for i in range(n):
        m = (masks[i] > 0.5).astype(np.uint8)

        # mask 해상도 보정 (예: 320 -> 원본 H,W)
        if m.shape[0] != H or m.shape[1] != W:
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)

        bb = bbox_from_mask(m)
        if bb is None:
            continue
        x1, y1, x2, y2 = bb
        cx = float((x1 + x2) / 2)
        cy = float((y1 + y2) / 2)

        area_px, peri_px, solidity, circularity = compute_basic_shape(m)

        # cm2 환산
        if (not np.isnan(cm_per_px)) and cm_per_px > 0:
            area_cm2 = float(area_px) * (cm_per_px ** 2)
        else:
            area_cm2 = np.nan

        row_flag = "t" if cy < (H / 2) else "b"

        instance_id = i + 1
        instance_uid = f"{bed_date}__{instance_id:03d}"

        bbox_w = float(x2 - x1)
        bbox_h = float(y2 - y1)
        aspect_ratio = float(bbox_w / bbox_h) if (np.isfinite(bbox_h) and bbox_h > 0) else np.nan

        front_height_cm = float(bbox_h * cm_per_px) if (np.isfinite(bbox_h) and np.isfinite(cm_per_px) and cm_per_px > 0) else np.nan

        brightness_mean = masked_brightness_mean_from_gray(gray, m)
        blur_score = masked_blur_score_from_lap(lap, m)

        rows.append({
            "bed_id": bed_id,
            "date": date,
            "bed_date": bed_date,
            "img_path": img_path,
            "warp_W": int(W),
            "warp_H": int(H),
            "instance_id": int(instance_id),
            "instance_uid": instance_uid,
            "conf": float(confs[i]),
            "centroid_x": float(cx),
            "centroid_y": float(cy),
            "x_norm": float(cx / W),
            "y_norm": float(cy / H),
            "bbox_x1": int(x1),
            "bbox_y1": int(y1),
            "bbox_x2": int(x2),
            "bbox_y2": int(y2),
            "bbox_w": float(bbox_w),
            "bbox_h": float(bbox_h),
            "aspect_ratio": float(aspect_ratio) if np.isfinite(aspect_ratio) else np.nan,
            "front_height_cm": float(front_height_cm) if np.isfinite(front_height_cm) else np.nan,
            "brightness_mean": float(brightness_mean) if np.isfinite(brightness_mean) else np.nan,
            "blur_score": float(blur_score) if np.isfinite(blur_score) else np.nan,
            "area_px": int(area_px),
            "area_front": int(area_px),
            "area_cm2": float(area_cm2) if np.isfinite(area_cm2) else np.nan,
            "perimeter_px": float(peri_px) if np.isfinite(peri_px) else np.nan,
            "solidity": float(solidity) if np.isfinite(solidity) else np.nan,
            "circularity": float(circularity) if np.isfinite(circularity) else np.nan,
            "touch_border": int(touch_border(m)),
            "row_flag": row_flag,
            "cm_per_px": float(cm_per_px) if np.isfinite(cm_per_px) else np.nan,
            "mm_per_px": float(cm_per_px * 10.0) if np.isfinite(cm_per_px) else np.nan,
            "px_per_mm": float(1.0 / (cm_per_px * 10.0)) if (np.isfinite(cm_per_px) and cm_per_px > 0) else np.nan,
        })

        mask_recs.append({
            "instance_uid": instance_uid,
            "bed_date": bed_date,
            "instance_id": int(instance_id),
            "rle": mask_to_rle(m),
        })

    return rows, mask_recs


In [ ]:

# =========================
# 5) Chunk 저장 / 재시작
# =========================

def load_done_index(out_dir: str, parquet_prefix: str):
    """chunk parquet들이 여러 개이므로, out_dir에서 prefix로 모두 찾아 bed_date를 모아 리턴"""
    pats = glob(os.path.join(out_dir, f"{parquet_prefix}_*.parquet"))
    if not pats:
        return set()
    done = set()
    for p in pats:
        try:
            df = pd.read_parquet(p, columns=["bed_date"])
            done |= set(df["bed_date"].dropna().unique().tolist())
        except Exception:
            continue
    return done
    try:
        df = pd.read_parquet(mid_parquet_path, columns=["bed_date"])
        return set(df["bed_date"].unique().tolist())
    except Exception:
        return set()

def append_jsonl(path: str, records: list):
    if not records:
        return
    with open(path, "a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def write_parquet_chunk(out_dir: str, parquet_prefix: str, chunk_idx: int, rows: list):
    """rows를 하나의 parquet chunk 파일로 저장 (append 위해 기존 파일 read/concat 하지 않음)"""
    if not rows:
        return None
    df = pd.DataFrame(rows)
    out_path = os.path.join(out_dir, f"{parquet_prefix}_{chunk_idx:05d}.parquet")
    df.to_parquet(out_path, index=False)
    return out_path

# =========================
# 6) Main
# =========================

def find_images_recursive(root: str):
    """root 하위 재귀 탐색으로 이미지 경로 리스트 반환"""
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    imgs = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(exts):
                imgs.append(os.path.join(dp, fn))
    imgs.sort()
    return imgs


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    tasks = find_images_recursive(BED_WARP_FOLDER)
    print(f"[{now_hms()}] [INFO] 이미지 개수: {len(tasks):,}")

    scale_map = load_scale_map(SCALE_CSV)
    print(f"[{now_hms()}] [INFO] scale_map 로드: {SCALE_CSV} (beds={len(scale_map)})")

    done_bed_dates = load_done_index(OUT_DIR, MID_PARQUET_PREFIX)
    if done_bed_dates:
        print(f"[{now_hms()}] [INFO] 이미 처리된 bed_date: {len(done_bed_dates):,}개")

    model = YOLO(SEG_MODEL_PATH)

    # bed_date 단위 스킵
    filtered = []
    for p in tasks:
        bed_id, date, bed_date = parse_bed_id_date(p)
        if bed_date is None:
            continue
        if bed_date in done_bed_dates:
            continue
        filtered.append(p)
    tasks = filtered
    print(f"[{now_hms()}] [INFO] 처리 대상(스킵 반영): {len(tasks):,}장")

    mask_jsonl_path = os.path.join(OUT_DIR, MASK_JSONL_NAME)

    buf_rows, buf_masks = [], []
    chunk_idx = 0

    t0 = time.time()

    for s in tqdm(range(0, len(tasks), BATCH_SIZE), desc="infer+mid", mininterval=0.5):
        batch_paths = tasks[s:s+BATCH_SIZE]

        imgs_bgr = load_images_batch(batch_paths, IO_WORKERS)
        valid = [(p, im) for p, im in zip(batch_paths, imgs_bgr) if im is not None]
        if not valid:
            continue

        v_paths = [p for p, _ in valid]
        v_bgrs  = [im for _, im in valid]
        v_rgbs  = [cv2.cvtColor(im, cv2.COLOR_BGR2RGB) for im in v_bgrs]

        results = model.predict(v_rgbs, conf=PRED_CONF, iou=PRED_IOU, verbose=False)

        for p, im_bgr, r0 in zip(v_paths, v_bgrs, results):
            rows, masks = process_one_result(r0, im_bgr, p, scale_map)
            buf_rows.extend(rows)
            buf_masks.extend(masks)

        if len(buf_rows) >= CHUNK_SIZE:
            write_parquet_chunk(OUT_DIR, MID_PARQUET_PREFIX, chunk_idx, buf_rows)
            append_jsonl(mask_jsonl_path, buf_masks)
            chunk_idx += 1
            buf_rows, buf_masks = [], []

    if buf_rows:
        write_parquet_chunk(OUT_DIR, MID_PARQUET_PREFIX, chunk_idx, buf_rows)
        append_jsonl(mask_jsonl_path, buf_masks)
        chunk_idx += 1

    dt = time.time() - t0
    print(f"[{now_hms()}] [DONE] mid-data 저장 완료: {OUT_DIR}/{MID_PARQUET_PREFIX}_*.parquet (chunks={chunk_idx}, sec={dt:.1f})")



In [ ]:

if __name__ == "__main__":
    main()

[16:26:37] [INFO] 이미지 개수: 2,192
[16:26:37] [INFO] scale_map 로드: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv (beds=96)
[16:26:37] [INFO] 처리 대상(스킵 반영): 2,192장


infer+mid: 100%|██████████| 137/137 [09:54<00:00,  4.34s/it]


[16:36:32] [DONE] mid-data 저장 완료: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/mid_instances_chunk_*.parquet (chunks=1, sec=595.4)


In [1]:
import pandas as pd

pq_path = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/mid_instances_chunk_00000.parquet"

df = pd.read_parquet(pq_path)

print("총 row 수:", len(df))
print("컬럼 수:", len(df.columns))
print("컬럼 목록:")
print(df.columns.tolist())

df.head()


총 row 수: 17344
컬럼 수: 34
컬럼 목록:
['bed_id', 'date', 'bed_date', 'img_path', 'warp_W', 'warp_H', 'instance_id', 'instance_uid', 'conf', 'centroid_x', 'centroid_y', 'x_norm', 'y_norm', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'bbox_w', 'bbox_h', 'aspect_ratio', 'front_height_cm', 'brightness_mean', 'blur_score', 'area_px', 'area_front', 'area_cm2', 'perimeter_px', 'solidity', 'circularity', 'touch_border', 'row_flag', 'cm_per_px', 'mm_per_px', 'px_per_mm']


,bed_id,date,bed_date,img_path,warp_W,warp_H,instance_id,instance_uid,conf,centroid_x,...,area_front,area_cm2,perimeter_px,solidity,circularity,touch_border,row_flag,cm_per_px,mm_per_px,px_per_mm
0,bed01,20251128,bed01_20251128,/content/drive/Othercomputers/내 컴퓨터/새 포...,1800,600,1,bed01_20251128__001,0.808197,1290.0,...,59382,809.393770,1455.386864,0.766981,0.352296,0,b,0.116749,1.167488,0.85654
1,bed01,20251128,bed01_20251128,/content/drive/Othercomputers/내 컴퓨터/새 포...,1800,600,2,bed01_20251128__002,0.557140,1006.0,...,60211,820.693279,1577.470124,0.748432,0.304063,1,b,0.116749,1.167488,0.85654
2,bed01,20251128,bed01_20251128,/content/drive/Othercomputers/내 컴퓨터/새 포...,1800,600,3,bed01_20251128__003,0.124302,1760.5,...,18114,246.899039,940.509666,0.823570,0.257334,1,b,0.116749,1.167488,0.85654
3,bed01,20251128,bed01_20251128,/content/drive/Othercomputers/내 컴퓨터/새 포...,1800,600,4,bed01_20251128__004,0.097941,221.5,...,87852,1197.448074,1607.068103,0.894268,0.427458,0,b,0.116749,1.167488,0.85654
4,bed01,20251128,bed01_20251128,/content/drive/Othercomputers/내 컴퓨터/새 포...,1800,600,5,bed01_20251128__005,0.078246,292.5,...,125152,1705.857821,2016.222431,0.863716,0.386875,0,b,0.116749,1.167488,0.85654


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17344 entries, 0 to 17343
Data columns (total 34 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bed_id           17344 non-null  object 
 1   date             17344 non-null  object 
 2   bed_date         17344 non-null  object 
 3   img_path         17344 non-null  object 
 4   warp_W           17344 non-null  int64  
 5   warp_H           17344 non-null  int64  
 6   instance_id      17344 non-null  int64  
 7   instance_uid     17344 non-null  object 
 8   conf             17344 non-null  float64
 9   centroid_x       17344 non-null  float64
 10  centroid_y       17344 non-null  float64
 11  x_norm           17344 non-null  float64
 12  y_norm           17344 non-null  float64
 13  bbox_x1          17344 non-null  int64  
 14  bbox_y1          17344 non-null  int64  
 15  bbox_x2          17344 non-null  int64  
 16  bbox_y2          17344 non-null  int64  
 17  bbox_w      

In [ ]:
# bed_date별 instance 개수
df.groupby("bed_date").size().describe()


,0
count,2192.000000
mean,7.912409
std,2.420359
min,1.000000
25%,6.000000
50%,8.000000
75%,9.000000
max,19.000000


In [ ]:
# 이미지당 instance 개수
df.groupby("img_path").size().describe()

,0
count,2192.000000
mean,7.912409
std,2.420359
min,1.000000
25%,6.000000
50%,8.000000
75%,9.000000
max,19.000000


In [ ]:
import json

jsonl_path = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/masks_rle.jsonl"

n_json = 0
with open(jsonl_path, "r") as f:
    for _ in f:
        n_json += 1

print("mask jsonl 개수:", n_json)
print("parquet row 수 :", len(df))
print("차이:", n_json - len(df))


mask jsonl 개수: 19352
parquet row 수 : 17344
차이: 2008


In [ ]:
df[["bed_date", "instance_id"]].duplicated().value_counts()

,count
False,17344
